In [2]:
import pandas as pd
from scipy.stats import bootstrap
import numpy as np

vision_base_eval = pd.read_csv("/Users/chandlershortlidge/Desktop/Ironhack/fitness-form-coach/llm-evals/vision-faithfulness-rag-hallucination-correctness-base.csv")

#### Scoring criteria: Correctness

- Score 0 — The generated answer contains incorrect or potentially harmful advice that contradicts the reference answer. This is the worst outcome.
- Score 1 — The generated answer is incomplete or omits important details from the reference, but contains nothing incorrect or harmful.
- Score 2 — The generated answer is correct and captures the key facts from the reference answer.

#### Scoring criteria: Hallucination

- Score 0 — The answer contains coaching advice that directly contradicts the retrieved context, or invents training recommendations with no basis in the context. This is the worst outcome.
- Score 1 — The coaching advice is mostly supported by the context but includes minor unsupported recommendations.
- Score 2 — All coaching advice and recommendations are directly supported by the retrieved context. Visual observations about the user's form do not count against this score.

In [3]:
vision_base_eval.head()

,id,inputs,reference_outputs,session_name,repetition,outputs,run,status,error,latency,tokens,total_cost,correctness,hallucination
0,2497560a-f99e-4d90-9374-928b4f5158be,"{""user_query"": ""are there any issues with my f...","{""answer"": ""The clearest visible issues are th...",vision-faithfulness-rag-v7-structured-output-4...,1,"{""answer"": ""Love the effort here—let’s dial a ...","{""inputs"": {""user_query"": ""are there any issue...",success,NaN,147.685615,20358,0.055014,1.0,1.0
1,289d6134-9559-4328-aa64-eac2084de0a0,"{""user_query"": ""How's my bench?""}","{""answer"": ""Slight movement in the feet. Wrist...",vision-faithfulness-rag-v7-structured-output-4...,1,"{""answer"": ""Love the outdoor grind—let’s dial ...","{""inputs"": {""user_query"": ""How's my bench?""}, ...",success,NaN,53.165919,10482,0.037487,0.5,2.0
2,308047c5-2ad2-4dfb-8c19-b983543ab83d,"{""user_query"": ""Can you check my form?""}","{""answer"": ""The biggest issue I can see is you...",vision-faithfulness-rag-v7-structured-output-4...,1,"{""answer"": ""Absolutely—thanks for the pics! He...","{""inputs"": {""user_query"": ""Can you check my fo...",success,NaN,62.943568,10241,0.033891,1.0,1.0
3,3273e894-50bb-46c3-a9e8-2d2831f40376,"{""user_query"": ""are there any issues with my f...","{""answer"": ""Your back looks completely flat on...",vision-faithfulness-rag-v7-structured-output-4...,1,"{""answer"": ""Awesome work getting full-range be...","{""inputs"": {""user_query"": ""are there any issue...",success,NaN,64.178148,15358,0.044209,0.5,1.0
4,58cbb032-850b-43f8-954b-f8ac3b9367f9,"{""user_query"": ""Are there any issues with my f...","{""answer"": ""The forearms do not appear vertica...",vision-faithfulness-rag-v7-structured-output-4...,1,"{""answer"": ""Awesome job getting after it. I wa...","{""inputs"": {""user_query"": ""Are there any issue...",success,NaN,54.787645,10332,0.034054,1.0,2.0


# Vision Base Eval: 10 Samples

In [4]:
# Find the standard deviation of the correctness and hallucination scores
vision_base_eval.describe()

,repetition,error,latency,tokens,total_cost,correctness,hallucination
count,10.0,0.0,10.000000,10.000000,10.000000,10.000000,10.000000
mean,1.0,NaN,67.350450,12951.800000,0.039158,0.750000,1.500000
std,0.0,NaN,32.852826,4344.870329,0.008369,0.263523,0.527046
min,1.0,NaN,42.945023,7923.000000,0.029911,0.500000,1.000000
25%,1.0,NaN,48.234488,10263.750000,0.033932,0.500000,1.000000
50%,1.0,NaN,53.976782,10423.000000,0.036674,0.750000,1.500000
75%,1.0,NaN,63.869503,15190.000000,0.043058,1.000000,2.000000
max,1.0,NaN,147.685615,20358.000000,0.055014,1.000000,2.000000


In [ ]:
# Find the 95% Confidence Interval, Standard Error, and Margin of Error for Correctness and Hallucination

def correctness_score(df, df_name):
    # 1. turn score into list for boostrap function
    correctness_array = df["correctness"].values
    #2. Use bootstrap function to dinf the Confidence Interval (default is 0.95) and standard error
    bootstrapped_hallucination = bootstrap(data=(correctness_array,), statistic=np.mean, method='BCa', rng=42)
    result = bootstrapped_hallucination
    print(f"95% CI Correctness Low - {df_name}: {result.confidence_interval.low}")
    print(f"95% CI Correctness High - {df_name}: {result.confidence_interval.high}")
    print(f"Standard Error - Correctness - {df_name}: {result.standard_error:.4f}")
    # 3. find the margin of error for Hallucination: MOE = (CI High - CI Low) / 2
    correctness_MOE = (result.confidence_interval.high - result.confidence_interval.low) / 2
    print(f"Correctness MOE - {df_name}: {correctness_MOE:.2f}")


def hallucination_score(df, df_name):
    # 1. turn score into list for boostrap function
    hallucination_array = df["hallucination"].values
    #2. Use bootstrap function to dinf the Confidence Interval (default is 0.95) and standard error
    bootstrapped_hallucination = bootstrap(data=(hallucination_array,), statistic=np.mean, method='BCa', rng=42)
    result = bootstrapped_hallucination
    print(f"95% CI Hallucination Low - {df_name}: {result.confidence_interval.low}")
    print(f"95% CI Hallucination High - {df_name}: {result.confidence_interval.high}")
    print(f"Standard Error - Hallucination - {df_name}: {result.standard_error:.4f}")
    # 3. find the margin of error for Hallucination: MOE = (CI High - CI Low) / 2
    hallucination_MOE = (result.confidence_interval.high - result.confidence_interval.low) / 2
    print(f"hallucination_MOE - {df_name}: {hallucination_MOE:.2f}")


In [45]:

base_10_hallucination = hallucination_score(vision_base_eval, df_name="vision_base_eval_10")
print(" ")
base_10_correctness = correctness_score(vision_base_eval, df_name="vision_base_eval_10")


95% CI Hallucination Low - vision_base_eval_10: 1.2
95% CI Hallucination High - vision_base_eval_10: 1.8
Standard Error - Hallucination - vision_base_eval_10: 0.1554
hallucination_MOE - vision_base_eval_10: 0.30
 
95% CI Correctness Low - vision_base_eval_10: 0.6
95% CI Correctness High - vision_base_eval_10: 0.9
Standard Error - Correctness - vision_base_eval_10: 0.0787
Correctness MOE - vision_base_eval_10: 0.15


#### Findings

- Score 1 — The coaching advice is mostly supported by the context but includes minor unsupported recommendations.
- Score 2 — All coaching advice and recommendations are directly supported by the retrieved context. Visual observations about the user's form do not count against this score.

Hallucination scores range between 1.2 and 1.8. This means they drift between containing some hallucinations (unsupported recommendations) and none. However, with n=10, it is impossible to say where the system truly sits between these scores.

#### Next steps

What happens if n=20, or n=30? Apply formula:
- new_MOE = old_MOE / sqrt(n2/n1)

In [8]:
hallucination_20_MOE = 0.3 / (np.sqrt(20/10))
print(f"MOE - 20 samples: {hallucination_20_MOE:.2f}")

hallucination_30_MOE = 0.3 / (np.sqrt(30/10))
print(f"MOE - 30 samples: {hallucination_30_MOE:.2f}")

MOE - 20 samples: 0.21
MOE - 30 samples: 0.17


# Vision Base Eval 20: test_1

In [10]:
vision_base_eval_20 = pd.read_csv("/Users/chandlershortlidge/Desktop/Ironhack/fitness-form-coach/llm-evals/vision-faithfulness-rag-20samples.csv")
vision_base_eval_20.head()

,id,inputs,reference_outputs,session_name,repetition,outputs,run,status,error,latency,tokens,total_cost,correctness,hallucination
0,1d422e61-d9b9-461d-9514-d167322c325a,"{""user_query"": ""How's my form?""}","{""answer"": ""Wrists are extended back under the...",vision-faithfulness-rag-v7-structured-output-4...,1,"{""answer"": ""Awesome work getting after it. Her...","{""inputs"": {""user_query"": ""How's my form?""}, ""...",success,NaN,42.583386,10285,0.031829,0.5,1.0
1,2497560a-f99e-4d90-9374-928b4f5158be,"{""user_query"": ""are there any issues with my f...","{""answer"": ""The clearest visible issues are th...",vision-faithfulness-rag-v7-structured-output-4...,1,"{""answer"": ""Love the effort here—let’s dial a ...","{""inputs"": {""user_query"": ""are there any issue...",success,NaN,42.086563,19562,0.045299,1.0,1.0
2,289d6134-9559-4328-aa64-eac2084de0a0,"{""user_query"": ""How's my bench?""}","{""answer"": ""Slight movement in the feet. Wrist...",vision-faithfulness-rag-v7-structured-output-4...,1,"{""answer"": ""Love the backyard grind! Thanks fo...","{""inputs"": {""user_query"": ""How's my bench?""}, ...",success,NaN,46.255542,10157,0.035699,0.5,1.0
3,308047c5-2ad2-4dfb-8c19-b983543ab83d,"{""user_query"": ""Can you check my form?""}","{""answer"": ""The biggest issue I can see is you...",vision-faithfulness-rag-v7-structured-output-4...,1,"{""answer"": ""Love that you filmed your bench—le...","{""inputs"": {""user_query"": ""Can you check my fo...",success,NaN,69.200048,10121,0.030731,1.0,2.0
4,3273e894-50bb-46c3-a9e8-2d2831f40376,"{""user_query"": ""are there any issues with my f...","{""answer"": ""Your back looks completely flat on...",vision-faithfulness-rag-v7-structured-output-4...,1,"{""answer"": ""Love that you filmed this—great st...","{""inputs"": {""user_query"": ""are there any issue...",success,NaN,45.183090,14805,0.039676,1.0,1.0


In [11]:
vision_base_eval_20.describe()

,repetition,error,latency,tokens,total_cost,correctness,hallucination
count,20.0,0.0,20.000000,20.000000,20.000000,20.00000,20.000000
mean,1.0,NaN,47.563381,13743.700000,0.037699,0.65000,1.600000
std,0.0,NaN,13.011740,4295.202343,0.005575,0.28562,0.502625
min,1.0,NaN,37.114877,8450.000000,0.028813,0.00000,1.000000
25%,1.0,NaN,41.421420,10228.500000,0.033708,0.50000,1.000000
50%,1.0,NaN,44.581522,10777.000000,0.036770,0.50000,2.000000
75%,1.0,NaN,47.141303,18730.000000,0.041026,1.00000,2.000000
max,1.0,NaN,94.627459,19562.000000,0.047601,1.00000,2.000000


In [46]:
base_10_hallucination = hallucination_score(vision_base_eval_20, df_name="vision_base_eval_20_test1")
print(" ")
base_10_correctness = correctness_score(vision_base_eval_20, df_name="vision_base_eval_20_test1")

95% CI Hallucination Low - vision_base_eval_20_test1: 1.4
95% CI Hallucination High - vision_base_eval_20_test1: 1.8
Standard Error - Hallucination - vision_base_eval_20_test1: 0.1081
hallucination_MOE - vision_base_eval_20_test1: 0.20
 
95% CI Correctness Low - vision_base_eval_20_test1: 0.525
95% CI Correctness High - vision_base_eval_20_test1: 0.775
Standard Error - Correctness - vision_base_eval_20_test1: 0.0634
Correctness MOE - vision_base_eval_20_test1: 0.12
